In [ ]:
# ==========================================
# 聚宽研究环境 TCN 二分类离线训练全管线
# ==========================================
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from torch.nn.utils import weight_norm
from jqdata import *
import warnings
warnings.filterwarnings('ignore')

# --- 参数设置 ---
START_DATE = '2019-01-01'
END_DATE = '2023-12-31'
SEQ_LENGTH = 30           # 时间窗长度
FUTURE_DAYS = 5           # 预测未来5天收益
BATCH_SIZE = 256          # 批处理大小
EPOCHS = 20               # 训练轮数
FEATURE_COLS = ['open', 'close', 'high', 'low', 'volume', 'money']

print("1. 开始获取股票池与历史数据...")
# 为了防止内存爆炸，我们选取中证1000成分股作为小盘股训练池代表
pool = get_index_stocks('000852.XSHG')[:200] # 取前200只做示范，实战可分批跑全量

df_price = get_price(pool, start_date=START_DATE, end_date=END_DATE, frequency='daily', fields=FEATURE_COLS, panel=False)
df_price = df_price.rename(columns={'time': 'date'})

print("2. 截面打标与特征标准化...")
# 计算未来5日收益率
df_price['future_ret'] = df_price.groupby('code')['close'].shift(-FUTURE_DAYS) / df_price['close'] - 1
df_price = df_price.dropna(subset=['future_ret'])

# 【核心动作】：截面二分类打标 (每天排名前 30% 为 1，其余为 0)
def assign_labels(group):
    group['label'] = (group['future_ret'].rank(pct=True) >= 0.70).astype(float)
    return group
df_price = df_price.groupby('date').apply(assign_labels).reset_index(drop=True)

# 时间序列 Z-score 标准化 (滚动60天，防止未来函数)
def normalize_features(group):
    roll_mean = group[FEATURE_COLS].rolling(60, min_periods=SEQ_LENGTH).mean()
    roll_std = group[FEATURE_COLS].rolling(60, min_periods=SEQ_LENGTH).std()
    group[FEATURE_COLS] = (group[FEATURE_COLS] - roll_mean) / (roll_std + 1e-8)
    return group
df_price = df_price.groupby('code').apply(normalize_features).dropna()

print("3. 滑动窗口构造 3D 张量...")
X_list, y_list = [], []

for code, group in df_price.groupby('code'):
    group = group.sort_values('date').reset_index(drop=True)
    features = group[FEATURE_COLS].values
    labels = group['label'].values
    
    num_samples = len(group) - SEQ_LENGTH + 1
    for i in range(num_samples):
        X_slice = features[i : i + SEQ_LENGTH, :] # 形状: (SEQ_LENGTH, FEATURES)
        y_label = labels[i + SEQ_LENGTH - 1]
        X_list.append(X_slice)
        y_list.append(y_label)

# 转换为 Numpy 数组
X_np = np.array(X_list)
y_np = np.array(y_list)

# 【关键】：PyTorch 的 Conv1d 要求输入形状为 (Batch, Channels, Length)
X_np = np.transpose(X_np, (0, 2, 1)) 

X_tensor = torch.tensor(X_np, dtype=torch.float32)
y_tensor = torch.tensor(y_np, dtype=torch.float32).unsqueeze(-1) # 形状 (N, 1)

print(f"张量构建完成！X 形状: {X_tensor.shape}, y 形状: {y_tensor.shape}")

# 构建 DataLoader
dataset = TensorDataset(X_tensor, y_tensor)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

# ======================== 4. 定义 TCN 模型 ========================
class ChainedCausalConv(nn.Module):
    def __init__(self, n_inputs, n_outputs, kernel_size, stride, dilation, padding, dropout=0.2):
        super(ChainedCausalConv, self).__init__()
        self.conv1 = weight_norm(nn.Conv1d(n_inputs, n_outputs, kernel_size, stride=stride, padding=padding, dilation=dilation))
        self.chomp1 = nn.ConstantPad1d((-padding, 0), 0)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)

        self.conv2 = weight_norm(nn.Conv1d(n_outputs, n_outputs, kernel_size, stride=stride, padding=padding, dilation=dilation))
        self.chomp2 = nn.ConstantPad1d((-padding, 0), 0)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)

        self.net = nn.Sequential(self.conv1, self.chomp1, self.relu1, self.dropout1,
                                 self.conv2, self.chomp2, self.relu2, self.dropout2)
        self.downsample = nn.Conv1d(n_inputs, n_outputs, 1) if n_inputs != n_outputs else None
        self.relu = nn.ReLU()

    def forward(self, x):
        out = self.net(x)
        res = x if self.downsample is None else self.downsample(x)
        return self.relu(out + res)

class TCNModel(nn.Module):
    def __init__(self, input_size, output_size, num_channels, kernel_size=3, dropout=0.2):
        super(TCNModel, self).__init__()
        layers = []
        num_levels = len(num_channels)
        for i in range(num_levels):
            dilation_size = 2 ** i
            in_channels = input_size if i == 0 else num_channels[i-1]
            out_channels = num_channels[i]
            layers += [ChainedCausalConv(in_channels, out_channels, kernel_size, stride=1, dilation=dilation_size,
                                         padding=(kernel_size-1) * dilation_size, dropout=dropout)]
        self.tcn = nn.Sequential(*layers)
        self.linear = nn.Linear(num_channels[-1], output_size)

    def forward(self, x):
        y1 = self.tcn(x)
        return self.linear(y1[:, :, -1])

# ======================== 5. 训练循环 ========================
print("4. 开始训练 TCN 模型...")
# 初始化模型 (6 个特征输入，输出 1 个 Logit 打分，3层残差网络)
model = TCNModel(input_size=len(FEATURE_COLS), output_size=1, num_channels=[16, 32, 64])

# 【核心】：使用带有 Sigmoid 结合的二元交叉熵损失函数
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

model.train()
for epoch in range(EPOCHS):
    total_loss = 0
    correct_preds = 0
    total_samples = 0
    
    for batch_X, batch_y in dataloader:
        optimizer.zero_grad()
        outputs = model(batch_X)
        
        # 计算损失
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        # 计算训练准确率 (Logit > 0 即代表概率 > 0.5)
        preds = (outputs > 0).float()
        correct_preds += (preds == batch_y).sum().item()
        total_samples += batch_y.size(0)
        
    avg_loss = total_loss / len(dataloader)
    accuracy = correct_preds / total_samples
    print(f"Epoch [{epoch+1}/{EPOCHS}], Loss: {avg_loss:.4f}, Accuracy: {accuracy:.4f}")

# ======================== 6. 保存模型权重 ========================
model_path = 'tcn_classification_weights.pth'
torch.save(model.state_dict(), model_path)
print(f"5. 训练完成！模型权重已保存至聚宽根目录: {model_path}")